# 02 — Conformal Hallucination Guardrail & Risk Control (`MAPIE`)

This notebook trains and evaluates base hallucination risk classifiers (`LogisticRegression` and `LightGBM`) on `data/processed/ragtruth_features.parquet`, calibrates split-conformal acceptance thresholds with `MAPIE` (`SplitConformalClassifier`, `conformity_score="lac"`) on the `cal` split, and inspects test-split reliability and risk-abstention trade-off curves.

## Control Objective
- **Pass / Abstain rule:** `PASS` when $\hat{p}(\text{hallucination}=1 \mid x) \le \text{threshold}(\alpha)$, else `ABSTAIN`.
- **Finite-sample guarantee:** $\mathbb{P}(Y = 1 \land \text{PASS}) \le \alpha$ across $\alpha \in [0.01, 0.30]$.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

from src.conformal.calibrator import (
    compute_ece,
    predict_risk_and_decision,
    run_conformal_pipeline,
)
from src.conformal.risk_model import RISK_FEATURE_COLS, predict_risk_proba

ROOT = Path("..") if Path("../data/processed/ragtruth_features.parquet").exists() else Path(".")
FEATURES_PARQUET = ROOT / "data/processed/ragtruth_features.parquet"
CONFIG_YAML = ROOT / "config/ragtruth_conformal.yaml"
ARTIFACTS_DIR = ROOT / "artifacts"

SWEEP_JSON = ARTIFACTS_DIR / "conformal_sweep.json"
METRICS_JSON = ARTIFACTS_DIR / "risk_metrics.json"
CALIBRATOR_PKL = ARTIFACTS_DIR / "calibrator.pkl"

if not (SWEEP_JSON.exists() and METRICS_JSON.exists() and CALIBRATOR_PKL.exists()):
    run_conformal_pipeline(FEATURES_PARQUET, CONFIG_YAML, ARTIFACTS_DIR)

sweep_data = json.loads(SWEEP_JSON.read_text(encoding="utf-8"))
risk_metrics = json.loads(METRICS_JSON.read_text(encoding="utf-8"))
with CALIBRATOR_PKL.open("rb") as fh:
    bundle = pickle.load(fh)

print(f"Status: {sweep_data['status']} | Primary model: {bundle.model_name}")
print(f"Splits: train={sweep_data['n_train']}, cal={sweep_data['n_cal']}, test={sweep_data['n_test']}")

## 1. Base Classifier Comparison & Benchmark Table (`RAGTruth` Test Split)

Comparing the uncalibrated cosine heuristic (`1 - cosine_sim <= 0.5`) against `MAPIE` split-conformalized `LogisticRegression` and `LightGBM`.

In [ ]:
rows = [
    risk_metrics["uncalibrated_heuristic"],
    risk_metrics["logistic_mapie"]["0.10"],
    risk_metrics["lightgbm_mapie"]["0.10"],
    risk_metrics["lightgbm_mapie"]["0.05"],
]
summary_df = pd.DataFrame(
    [
        {
            "Guardrail Method": r["guardrail_method"],
            "Target alpha": "N/A" if r["target_alpha"] is None else f"{r['target_alpha']:.2f}",
            "Threshold": round(r["threshold"], 4),
            "Empirical False-Accept Rate": round(r["false_accept_rate"], 4),
            "Abstention Rate": round(r["abstention_rate"], 4),
            "ECE": round(r["ece"], 4),
            "Brier Score": round(r["brier_score"], 4),
        }
        for r in rows
    ]
)
display(summary_df)

## 2. Calibration Reliability (ECE / Brier Score on Test Split)

In [ ]:
df_test = pd.read_parquet(FEATURES_PARQUET)
df_test = df_test[df_test["split"] == "test"].reset_index(drop=True)
X_test = df_test[RISK_FEATURE_COLS].to_numpy(dtype=float)
y_test = df_test["has_hallucination"].astype(int).to_numpy()
probs_test = predict_risk_proba(bundle.estimator, X_test)

bin_edges = np.linspace(0.0, 1.0, 11)
cal_rows = []
for idx in range(10):
    lo, hi = bin_edges[idx], bin_edges[idx + 1]
    mask = (probs_test >= lo) & (probs_test <= hi) if idx == 0 else (probs_test > lo) & (probs_test <= hi)
    if np.any(mask):
        cal_rows.append(
            {
                "bin": f"({lo:.1f}, {hi:.1f}]",
                "count": int(np.sum(mask)),
                "mean_pred_prob": round(float(np.mean(probs_test[mask])), 4),
                "observed_halluc_rate": round(float(np.mean(y_test[mask])), 4),
            }
        )
print(f"Test ECE (10 bins): {compute_ece(y_test, probs_test):.4f}")
display(pd.DataFrame(cal_rows))

## 3. MAPIE Risk-Abstention Trade-off Curves ($\alpha \in [0.01, 0.30]$)

In [ ]:
lgb_df = pd.DataFrame(sweep_data["sweep"])
log_df = pd.DataFrame(sweep_data["logistic_sweep"])

def render_tradeoff_svg(lgb: pd.DataFrame, log: pd.DataFrame) -> HTML:
    w, h, pad = 560, 260, 42
    pw, ph = w - 2 * pad, h - 2 * pad

    def pts(df_in: pd.DataFrame, col: str, max_y: float) -> str:
        coords = []
        for _, row in df_in.iterrows():
            x = pad + (row["alpha"] - 0.01) / 0.29 * pw
            y = h - pad - min(1.0, row[col] / max_y) * ph
            coords.append(f"{x:.1f},{y:.1f}")
        return " ".join(coords)

    diag = f"{pad},{h - pad} {pad + pw},{h - pad - (0.30 / 0.35) * ph}"
    svg_far = (
        f'<svg width="{w}" height="{h}" style="border:1px solid #e2e8f0;background:#fff;">'
        f'<polyline fill="none" stroke="#94a3b8" stroke-dasharray="4,4" stroke-width="1.5" points="{diag}"/>'
        f'<polyline fill="none" stroke="#0284c7" stroke-width="2" points="{pts(log, "false_accept_rate", 0.35)}"/>'
        f'<polyline fill="none" stroke="#0f766e" stroke-width="2.5" points="{pts(lgb, "false_accept_rate", 0.35)}"/>'
        f'<text x="{pad}" y="20" font-size="12" font-family="monospace">'
        f'False-Accept Rate vs Target alpha (dashed=y=alpha, teal=LightGBM, blue=Logistic)</text>'
        f'</svg>'
    )
    svg_abs = (
        f'<svg width="{w}" height="{h}" style="border:1px solid #e2e8f0;background:#fff;margin-top:10px;">'
        f'<polyline fill="none" stroke="#0284c7" stroke-width="2" points="{pts(log, "abstention_rate", 1.0)}"/>'
        f'<polyline fill="none" stroke="#0f766e" stroke-width="2.5" points="{pts(lgb, "abstention_rate", 1.0)}"/>'
        f'<text x="{pad}" y="20" font-size="12" font-family="monospace">'
        f'Abstention Rate vs Target alpha (teal=LightGBM, blue=Logistic)</text>'
        f'</svg>'
    )
    return HTML(svg_far + "<br/>" + svg_abs)

display(render_tradeoff_svg(lgb_df, log_df))
display(lgb_df[lgb_df["alpha"].isin([0.01, 0.05, 0.10, 0.15, 0.20, 0.30])].round(4))

sample_row = df_test.iloc[[0]][RISK_FEATURE_COLS]
risk, decision, threshold = predict_risk_and_decision(bundle, sample_row, alpha=0.10)
print(f"Sample test[0] @ alpha=0.10 -> risk={risk:.4f}, threshold={threshold:.4f}, decision={decision}")